# Second Pass — Fine-tune SportsBERT NER

Fine-tunes `microsoft/SportsBERT` on the auto-labelled dataset produced by `initial_generation.ipynb`.

**Why:** The initial labels were produced by a generic BERT-NER model with no sports context. Fine-tuning on those labels with a sports-domain base model yields better precision on football-specific injury and status language (e.g. "ruled out", "doubtful for Saturday", "hamstring complaint").

**Input:** `data/initial_labels.jsonl` — BIO-tagged sentences (`B-PLAYER / I-PLAYER / B-INJURY / I-INJURY / B-STATUS / I-STATUS / O`)  
**Output:** `models/sports-ner/best` — fine-tuned model ready for inference over the article cache

## Config

In [ ]:
from pathlib import Path

BASE_MODEL = "microsoft/SportsBERT"
DATA_PATH  = Path("../../data/initial_labels.jsonl")
OUT_DIR    = Path("../../models/sports-ner")

LABEL_LIST = ["O", "B-PLAYER", "I-PLAYER", "B-INJURY", "I-INJURY", "B-STATUS", "I-STATUS"]
LABEL2ID   = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL   = {i: l for l, i in LABEL2ID.items()}

## Load initial labels

Reads `initial_labels.jsonl` and splits into train / eval (90/10 random split). Temporal ordering does not matter here since these are individual sentences, not match sequences.

In [ ]:
import json
import numpy as np

rows = [json.loads(l) for l in open(DATA_PATH)]
print(f"Total sentences: {len(rows):,}")

rng = np.random.default_rng(42)
idx = rng.permutation(len(rows))
cut = int(len(rows) * 0.9)
train_rows = [rows[i] for i in idx[:cut]]
eval_rows  = [rows[i] for i in idx[cut:]]
print(f"Train: {len(train_rows):,}  |  Eval: {len(eval_rows):,}")

## Tokenise and align labels

WordPiece tokenisation splits words into subword tokens, breaking the 1-to-1 alignment between words and BIO tags. The helper below:
1. Tokenises pre-split token lists with `is_split_into_words=True`
2. Uses `word_ids()` to map each subword back to its source word
3. Assigns `B-` to the first subword, converts to matching `I-` for continuations, and masks special tokens with `-100` so they are ignored by the loss

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=512,
    )
    aligned_labels = []
    for i, tags in enumerate(examples["tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(LABEL2ID[tags[word_id]])
            else:
                tag = tags[word_id]
                if tag.startswith("B-"):
                    tag = "I-" + tag[2:]
                label_ids.append(LABEL2ID.get(tag, -100))
            prev_word_id = word_id
        aligned_labels.append(label_ids)
    tokenized["labels"] = aligned_labels
    return tokenized

def rows_to_dataset(rows):
    d = {"tokens": [r["tokens"] for r in rows], "tags": [r["tags"] for r in rows]}
    ds = Dataset.from_dict(d)
    return ds.map(tokenize_and_align, batched=True, remove_columns=["tokens", "tags"])

train_ds = rows_to_dataset(train_rows)
eval_ds  = rows_to_dataset(eval_rows)
print(train_ds)

## Load SportsBERT with NER head

`microsoft/SportsBERT` is a masked LM with no downstream head, so `ignore_mismatched_sizes=True` drops the LM head and replaces it with a randomly-initialised token classification layer sized to our 7-class schema.

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

## Evaluation metrics

Uses `seqeval` for span-level F1 — the standard NER metric. A span only counts as correct if both the full entity boundary and label match, which is stricter than token-level accuracy.

In [ ]:
from seqeval.metrics import classification_report

def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    true_preds, true_labels = [], []
    for pred_seq, label_seq in zip(preds, labels):
        tp, tl = [], []
        for p_id, l_id in zip(pred_seq, label_seq):
            if l_id != -100:
                tp.append(ID2LABEL[p_id])
                tl.append(ID2LABEL[l_id])
        true_preds.append(tp)
        true_labels.append(tl)
    report = classification_report(true_labels, true_preds, output_dict=True)
    return {
        "f1":        report["weighted avg"]["f1-score"],
        "precision": report["weighted avg"]["precision"],
        "recall":    report["weighted avg"]["recall"],
    }

## Train

3 epochs is sufficient for fine-tuning BERT on a labelled NER dataset of this size. `load_best_model_at_end=True` restores the checkpoint with the highest eval F1 before saving.

Set `fp16=False` if running on CPU or Apple Silicon (MPS does not support fp16 for all ops).

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

args = TrainingArguments(
    output_dir=str(OUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    logging_steps=500,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

## Save best model

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUT_DIR / "best"))
tokenizer.save_pretrained(str(OUT_DIR / "best"))
print("Model saved to", OUT_DIR / "best")

## Sanity check predictions

Runs the saved model on a handful of eval sentences and prints predicted tags alongside ground truth. Useful for spotting systematic errors before running inference over the full article cache.

In [ ]:
from transformers import pipeline

ner_pipe = pipeline(
    "ner",
    model=str(OUT_DIR / "best"),
    aggregation_strategy="simple",
)

for row in eval_rows[:5]:
    sentence = row["sentence"]
    preds = ner_pipe(sentence)
    gold  = [(t, g) for t, g in zip(row["tokens"], row["tags"]) if g != "O"]
    print("SENTENCE:", sentence)
    print("GOLD:    ", gold)
    print("PRED:    ", [(p["word"], p["entity_group"]) for p in preds])
    print()